# JobInsight-104 — Google Colab 版本

104.com.tw 職缺爬蟲與分析平台，在 Colab 上一鍵啟動。

**流程：**
1. Clone 專案
2. 安裝 Python 套件
3. 建置前端（React → 靜態檔案）
4. 修改後端以提供靜態檔案
5. 啟動伺服器 + 取得臨時公開網址

> 依序執行所有 cell，或選擇「執行階段 → 全部執行」

## 1. Clone 專案

In [ ]:
import os

!git clone https://github.com/kobojp/JobInsight-104.git
os.chdir('/content/JobInsight-104')
print('Working directory:', os.getcwd())

## 2. 安裝 Python 套件

In [ ]:
!pip install -q \
    fastapi \
    "uvicorn[standard]" \
    pydantic \
    pandas \
    requests \
    python-dotenv

print('Python 套件安裝完成')

## 3. 建置前端

`npm run build` 會將 React 編譯為靜態檔案，輸出至 `frontend/dist/`。

In [ ]:
%%bash
cd /content/JobInsight-104/frontend
npm install --silent
npm run build
echo "前端建置完成，輸出目錄："
ls dist/

## 4. 修改後端：加入靜態檔案服務

原本的 `backend/main.py` 只提供 API。
此 cell 在執行時期注入靜態檔案掛載與 SPA catch-all 路由，
讓同一個 FastAPI 伺服器同時服務前端與 API。

In [ ]:
PATCHED_MAIN = '''
"""
JobInsight-104 FastAPI backend — Colab 版（含靜態檔案服務）
"""

from contextlib import asynccontextmanager
from pathlib import Path

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse

from backend.database import init_db
from backend.routes.search import router as search_router
from backend.routes.tasks import router as tasks_router
from backend.routes.export import router as export_router

DIST_DIR = Path(__file__).parent.parent / "frontend" / "dist"


@asynccontextmanager
async def lifespan(app: FastAPI):
    init_db()
    yield


app = FastAPI(
    title="JobInsight-104 API",
    version="1.0.0",
    lifespan=lifespan,
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# API 路由
app.include_router(search_router, prefix="/api")
app.include_router(tasks_router, prefix="/api")
app.include_router(export_router, prefix="/api")


@app.get("/api/health")
async def health():
    return {"status": "ok", "version": "1.0.0"}


# 靜態資源（JS/CSS/圖片）
if (DIST_DIR / "assets").exists():
    app.mount("/assets", StaticFiles(directory=str(DIST_DIR / "assets")), name="assets")


# SPA catch-all：其他所有路徑都回傳 index.html
@app.get("/{full_path:path}")
async def serve_spa(full_path: str):
    index = DIST_DIR / "index.html"
    if index.exists():
        return FileResponse(str(index))
    return {"error": "Frontend not built. Run: cd frontend && npm run build"}
'''

with open('/content/JobInsight-104/backend/main.py', 'w', encoding='utf-8') as f:
    f.write(PATCHED_MAIN)

print('backend/main.py 已更新（加入靜態檔案服務）')

## 5. 安裝 cloudflared（免費臨時網址，不需帳號）

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
     -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

## 6. 啟動伺服器並取得公開網址

- FastAPI 在背景執行（port 8000）
- cloudflared 建立 tunnel，輸出 `https://xxxx.trycloudflare.com`

> 網址出現後即可點擊使用。每次重啟 Colab 網址都會改變。

In [ ]:
import subprocess
import threading
import time
import re
import sys

# ── 啟動 uvicorn（背景執行緒）──────────────────────────────────
def run_server():
    subprocess.run(
        [sys.executable, '-m', 'uvicorn', 'backend.main:app',
         '--host', '0.0.0.0', '--port', '8000'],
        cwd='/content/JobInsight-104'
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print('uvicorn 啟動中...')
time.sleep(3)

# ── 健康檢查 ───────────────────────────────────────────────────
import urllib.request
try:
    with urllib.request.urlopen('http://localhost:8000/api/health', timeout=5) as r:
        print('後端狀態:', r.read().decode())
except Exception as e:
    print('後端未就緒，請稍候重試：', e)

# ── 啟動 cloudflared tunnel ────────────────────────────────────
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print('等待 cloudflared 建立 tunnel...')
public_url = None
for line in cf_proc.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group()
        break

if public_url:
    print()
    print('=' * 55)
    print(f'  公開網址: {public_url}')
    print('=' * 55)
    print('  點擊網址即可開啟 JobInsight-104 介面')
    print('  (關閉 Colab 後網址失效)')
else:
    print('無法取得 tunnel 網址，請檢查 cloudflared 輸出')

---

## 使用說明

### 搜尋職缺
1. 在介面輸入關鍵字（例：Python、數據分析）
2. 選擇地區與排序方式
3. 點擊「搜尋」，即時顯示爬取進度
4. 完成後可查看統計圖表、篩選結果、匯出 CSV

### 注意事項
- 每次重啟 Colab runtime，需重新執行所有 cell
- 臨時網址每次不同，無法固定
- 爬蟲每頁請求間隔 3–6 秒（避免被封鎖）
- 搜尋歷史儲存於 Colab 本機 SQLite，關閉 runtime 後消失

### CLI 模式（可選）
```python
# 直接在 Colab cell 執行 CLI 爬蟲
!cd /content/JobInsight-104 && python main.py -k "Python" -a taipei -n 3
```

---

## （選用）CLI 單次爬取 + 下載結果

In [ ]:
# 修改以下參數後執行
KEYWORD = 'Python'     # 搜尋關鍵字
AREA    = 'taipei'     # 地區（參考 config.py 的 AREA_CODES）
PAGES   = 3            # 最大頁數
FORMAT  = 'csv'        # 'csv' / 'json' / 'both'

!mkdir -p /content/JobInsight-104/output
!cd /content/JobInsight-104 && \
    python main.py -k "{KEYWORD}" -a {AREA} -n {PAGES} --format {FORMAT}

# 列出輸出檔案
import glob
files = glob.glob('/content/JobInsight-104/output/*')
print('\n輸出檔案：')
for f in sorted(files):
    print(' ', f)

In [ ]:
# 下載最新的 CSV 到本機
import glob
from google.colab import files

csv_files = sorted(glob.glob('/content/JobInsight-104/output/*.csv'))
if csv_files:
    latest = csv_files[-1]
    print(f'下載：{latest}')
    files.download(latest)
else:
    print('尚未有 CSV 輸出，請先執行上方 CLI cell')